> Adapted from *Dive into Deep Learning*, section [Concise Implementation of Recurrent Neural Networks](https://d2l.ai/chapter_recurrent-neural-networks/rnn-concise.html). Not affiliated with or endorsed by the book's authors.
> Source, authors, licenses and changes: [Attribution and changes](#Attribution-and-changes), at the end of this notebook.

Previous: [Section 9.5: Recurrent Neural Network Implementation from Scratch](09.05_rnn-scratch.ipynb) | Next: [Section 9.7: Backpropagation Through Time](09.07_bptt.ipynb)

# Concise Implementation of Recurrent Neural Networks

Like most of our from-scratch implementations,
[Section 9.5](https://d2l.ai/chapter_recurrent-neural-networks/rnn-scratch.html#sec-rnn-scratch) was designed 
to provide insight into how each component works.
But when you are using RNNs every day 
or writing production code,
you will want to rely more on libraries
that cut down on both implementation time 
(by supplying library code for common models and functions)
and computation time 
(by optimizing the heck out of these library implementations).
This section will show you how to implement 
the same language model more efficiently
using the high-level API provided 
by your deep learning framework.
We begin, as before, by loading 
*The Time Machine* dataset.

In [ ]:
from divelet_ml import nn
import autograd.numpy as np

import divelet_ml as d2l

## Defining the Model

We define the following class
using the RNN implemented
by high-level APIs.

In [ ]:
class RNN(d2l.Module):  #@save
    """The RNN model implemented with high-level APIs."""
    def __init__(self, num_inputs, num_hiddens):
        super().__init__()
        self.save_hyperparameters()
        self.rnn = nn.RNN(num_inputs, num_hiddens)

    def forward(self, inputs, H=None):
        return self.rnn(inputs, H)

Inheriting from the `RNNLMScratch` class in [Section 9.5](https://d2l.ai/chapter_recurrent-neural-networks/rnn-scratch.html#sec-rnn-scratch), 
the following `RNNLM` class defines a complete RNN-based language model.
Note that we need to create a separate fully connected output layer.

In [ ]:
class RNNLM(d2l.RNNLMScratch):  #@save
    """The RNN-based language model implemented with high-level APIs."""
    def init_params(self):
        self.linear = nn.LazyLinear(self.vocab_size)

    def output_layer(self, hiddens):
        return np.swapaxes(self.linear(hiddens), 0, 1)

## Training and Predicting

Before training the model, let's make a prediction 
with a model initialized with random weights.
Given that we have not trained the network, 
it will generate nonsensical predictions.

In [ ]:
data = d2l.TimeMachine(batch_size=1024, num_steps=32)
rnn = RNN(num_inputs=len(data.vocab), num_hiddens=32)
model = RNNLM(rnn, vocab_size=len(data.vocab), lr=1)
model.predict('it has', 20, data.vocab)

Next, we train our model, leveraging the high-level API.

In [ ]:
trainer = d2l.Trainer(max_epochs=d2l.browser_epochs(100, pyodide=25, xeus=50), gradient_clip_val=1)
trainer.fit(model, data)

<img src="../tablet.svg" alt="divelet-ml" title="Added by divelet-ml, not in the book"> **Full run, for comparison.** In JupyterLite, the cell above trains less than the book does (see the note before it). This figure is what it gives with 100 epochs, as in the book: computed with CPython, not in the browser, from a random start (seed 0), so a run of your own will differ in the details. Final values: train_ppl 5.944, val_ppl 7.479.

Training curves of the full run. Final values: train_ppl 5.944, val_ppl 7.479.

Compared with [Section 9.5](https://d2l.ai/chapter_recurrent-neural-networks/rnn-scratch.html#sec-rnn-scratch),
this model achieves comparable perplexity,
but runs faster due to the optimized implementations.
As before, we can generate predicted tokens 
following the specified prefix string.

<img src="../tablet.svg" alt="divelet-ml" title="Added by divelet-ml, not in the book"> Unlike in the book, `nn.RNN` is not faster than the from-scratch model here: divelet-ml's `nn.RNN` is also a Python loop over the time steps, so the two take about the same time, and the perplexity is the same kind.

In [ ]:
model.predict('it has', 20, data.vocab)

## Summary

High-level APIs in deep learning frameworks provide implementations of standard RNNs.
These libraries help you to avoid wasting time reimplementing standard models.
Moreover,
framework implementations are often highly optimized, 
  leading to significant (computational) performance gains 
  when compared with implementations from scratch.

## Exercises

1. Can you make the RNN model overfit using the high-level APIs?
1. Implement the autoregressive model of [Section 9.1](https://d2l.ai/chapter_recurrent-neural-networks/sequence.html#sec-sequence) using an RNN.

Discussions of the original section: [book's forum](https://discuss.d2l.ai/t/1053)

Previous: [Section 9.5: Recurrent Neural Network Implementation from Scratch](09.05_rnn-scratch.ipynb) | Next: [Section 9.7: Backpropagation Through Time](09.07_bptt.ipynb)

## Attribution and changes

Adapted from *Dive into Deep Learning*, section
[Concise Implementation of Recurrent Neural Networks](https://d2l.ai/chapter_recurrent-neural-networks/rnn-concise.html)
(source: [`chapter_recurrent-neural-networks/rnn-concise.md`](https://github.com/d2l-ai/d2l-en/blob/23d7a5aecceee57d1292c56e90cce307f183bb0a/chapter_recurrent-neural-networks/rnn-concise.md)
at commit `23d7a5aeccee`), by Aston Zhang, Zachary C. Lipton, Mu Li, Alexander J. Smola and contributors.
Text: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). Code: adapted from the book's sample code, MIT-0.
Not affiliated with or endorsed by the book's authors.

**Changes from the original** (PyTorch edition):
- Code uses NumPy instead of PyTorch, so it runs in JupyterLite (Pyodide and xeus-python).
- `d2l` is divelet-ml's package (`import divelet_ml as d2l`) instead of the book's `d2l.torch`. `%matplotlib inline` is removed: both JupyterLite kernels display plots inline.
- Training for 100 epochs takes minutes in the browser, so `d2l.browser_epochs(100, pyodide=25, xeus=50)` gives 100 epochs outside JupyterLite but 50 in xeus-python and 25 in Pyodide, whose NumPy has no BLAS. This is not in the book: in the browser, the curves show less training, and the perplexity is higher and the generated text worse than the book's. Set `max_epochs=100` to train as long as the book does.
- `nn` is divelet-ml's `divelet_ml.nn`: its `nn.RNN` is a NumPy stand-in for `torch.nn.RNN`, a one-layer tanh network with the same inputs and outputs, written as a Python loop over the time steps that autograd differentiates. So it is not faster than the from-scratch model of 9.5 here, unlike the book's PyTorch version.
- `d2l.swapaxes` becomes `np.swapaxes`; `d2l.Trainer(..., num_gpus=1)`, `d2l.try_gpu()` and the `device` argument of `predict` are removed: a browser kernel has no GPU.
- Equations keep the book's numbers. Cross-references to other sections link to the book's site, and citations are written inline (author and year); the book's bibliography lists them.
- The d2lbook framework tabs for MXNet, TensorFlow and JAX are removed.
- After each training cell, a figure marked with the tablet shows what the book's full training gives, computed with CPython, not in the browser (made by `tools/full_run_figures.py`). It is not in the book.
